# AT1 - Perceptron e KNN em Prática

Trabalho prático individual: implementação do **Perceptron com Treinamento Automático**, do **Classificador KNN** e de um **Recomendador por Similaridade Espacial**, utilizando NumPy de forma vetorizada.

In [1]:
import numpy as np

## Desafio 1: Classificação Binária com Perceptron Treinável

Detecção preliminar de transações suspeitas a partir de duas métricas: valor normalizado da transação e frequência de operações recentes.

In [2]:
# Matriz de treino: [Valor Normalizado (0.0 a 10.0), Frequência de Operações]
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

# Rótulos das classes (0: Legítima, 1: Suspeita)
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

In [3]:
def funcao_degrau(z):
    """Função de ativação degrau: retorna 1 se z >= 0, senão 0."""
    return np.where(z >= 0, 1, 0)


def treinar_perceptron(X, y, taxa_aprendizado, epocas):
    """Treina um Perceptron com a regra de Rosenblatt.

    Pesos e viés são inicializados com 1.0, conforme especificação.
    Retorna os pesos (w) e o viés (b) ajustados.
    """
    n_amostras, n_features = X.shape
    w = np.ones(n_features)
    b = 1.0

    for _ in range(epocas):
        for i in range(n_amostras):
            z = np.dot(X[i], w) + b
            y_pred = funcao_degrau(z)
            erro = y[i] - y_pred
            if erro != 0:
                w += taxa_aprendizado * erro * X[i]
                b += taxa_aprendizado * erro

    return w, b


def prever_perceptron(amostra, w, b):
    """Aplica a combinação linear e a função degrau para prever a classe (0 ou 1)."""
    z = np.dot(amostra, w) + b
    return funcao_degrau(z)

In [4]:
taxa_aprendizado = 0.1
epocas = 50

w_final, b_final = treinar_perceptron(X_treino_fraude, y_treino_fraude, taxa_aprendizado, epocas)

print(f"Pesos finais (w): {w_final}")
print(f"Viés final (b): {b_final}")

transacao_a = np.array([5.0, 4.0])
transacao_b = np.array([7.0, 6.5])

classe_a = prever_perceptron(transacao_a, w_final, b_final)
classe_b = prever_perceptron(transacao_b, w_final, b_final)

rotulos = {0: "Transação Legítima", 1: "Transação Suspeita (Risco de Fraude)"}

print(f"\nTransação A {transacao_a} -> {rotulos[int(classe_a)]}")
print(f"Transação B {transacao_b} -> {rotulos[int(classe_b)]}")

Pesos finais (w): [-5.55111512e-16  4.00000000e-01]
Viés final (b): -1.2

Transação A [5. 4.] -> Transação Suspeita (Risco de Fraude)
Transação B [7.  6.5] -> Transação Suspeita (Risco de Fraude)


## Desafio 2: Predição de Risco de Churn com Classificador KNN

Antecipação de risco de cancelamento de clientes SaaS a partir de dias de inatividade e chamados críticos em aberto.

In [5]:
# Matriz de treino: [Dias de Inatividade, Chamados Críticos]
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

# Rótulos (0: Baixo Risco, 1: Alto Risco)
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

In [6]:
def classificar_knn(X_treino, y_treino, ponto, k, metrica="euclidiana"):
    """Classifica um ponto por votação majoritária dos k vizinhos mais próximos.

    As distâncias entre o ponto e todas as amostras de treino são calculadas
    de forma vetorizada (sem laços manuais).
    Retorna a classe predita, os índices dos vizinhos e as distâncias correspondentes.
    """
    diffs = X_treino - ponto

    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum(diffs ** 2, axis=1))
    elif metrica == "manhattan":
        distancias = np.sum(np.abs(diffs), axis=1)
    else:
        raise ValueError("Métrica não suportada. Use 'euclidiana' ou 'manhattan'.")

    indices_ordenados = np.argsort(distancias)
    indices_vizinhos = indices_ordenados[:k]
    distancias_vizinhos = distancias[indices_vizinhos]

    classes_vizinhos = y_treino[indices_vizinhos]
    valores, contagens = np.unique(classes_vizinhos, return_counts=True)
    classe_predita = valores[np.argmax(contagens)]

    return classe_predita, indices_vizinhos, distancias_vizinhos

In [7]:
rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}

cliente_1 = np.array([13.0, 10.0])
k1 = 3
classe_1, indices_1, distancias_1 = classificar_knn(X_treino_churn, y_treino_churn, cliente_1, k1, "euclidiana")

print(f"Cliente 1 {cliente_1} (k={k1})")
print(f"Classe predita: {rotulos_churn[int(classe_1)]}")
print(f"Índices dos vizinhos: {indices_1}")
print(f"Distâncias dos vizinhos: {distancias_1}")

cliente_2 = np.array([7.0, 1.5])
k2 = 5
classe_2, indices_2, distancias_2 = classificar_knn(X_treino_churn, y_treino_churn, cliente_2, k2, "euclidiana")

print(f"\nCliente 2 {cliente_2} (k={k2})")
print(f"Classe predita: {rotulos_churn[int(classe_2)]}")
print(f"Índices dos vizinhos: {indices_2}")
print(f"Distâncias dos vizinhos: {distancias_2}")

Cliente 1 [13. 10.] (k=3)
Classe predita: Alto Risco
Índices dos vizinhos: [4 5 6]
Distâncias dos vizinhos: [6.32455532 7.07106781 9.21954446]

Cliente 2 [7.  1.5] (k=5)
Classe predita: Baixo Risco
Índices dos vizinhos: [3 2 1 0 4]
Distâncias dos vizinhos: [1.11803399 2.06155281 4.27200187 5.02493781 8.38152731]


## Desafio 3: Recomendação de Servidores Cloud por Similaridade Espacial

Sistema de dimensionamento automático que recomenda as VMs do catálogo mais próximas (distância euclidiana) do perfil de hardware demandado.

In [8]:
# Catálogo: [vCPUs, Memória RAM (GB), Armazenamento SSD (GB)]
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

In [9]:
def recomendar_servidores(catalogo, perfil_demandado, k):
    """Recomenda os k servidores mais próximos do perfil demandado (distância euclidiana).

    Retorna os índices ordenados por proximidade e as respectivas distâncias.
    """
    diffs = catalogo - perfil_demandado
    distancias = np.sqrt(np.sum(diffs ** 2, axis=1))

    indices_ordenados = np.argsort(distancias)[:k]
    distancias_ordenadas = distancias[indices_ordenados]

    return indices_ordenados, distancias_ordenadas

In [10]:
perfil_demandado = np.array([12.0, 28.0, 850.0])
k_recomendacoes = 2

indices_recomendados, distancias_recomendadas = recomendar_servidores(
    catalogo_servidores, perfil_demandado, k_recomendacoes
)

print(f"Perfil de Hardware Demandado: {perfil_demandado}\n")
print("Relatório de Recomendação de Servidores")
print("=" * 45)

for posicao, (idx, dist) in enumerate(zip(indices_recomendados, distancias_recomendadas), start=1):
    vcpus, ram, ssd = catalogo_servidores[idx]
    print(f"{posicao}º lugar: {nomes_servidores[idx]}")
    print(f"  Especificações: {vcpus:.0f} vCPUs, {ram:.0f} GB RAM, {ssd:.0f} GB SSD")
    print(f"  Distância euclidiana: {dist:.2f}")
    print("-" * 45)

Perfil de Hardware Demandado: [ 12.  28. 850.]

Relatório de Recomendação de Servidores
1º lugar: High Performance Computing
  Especificações: 32 vCPUs, 64 GB RAM, 1000 GB SSD
  Distância euclidiana: 155.55
---------------------------------------------
2º lugar: Database Enterprise
  Especificações: 16 vCPUs, 32 GB RAM, 500 GB SSD
  Distância euclidiana: 350.05
---------------------------------------------
